# CC0F4 - Tercera Práctica Calificada
## De un sistema RAG evaluable a un workflow, un agente controlado y una memoria verificable

**Estudiante:** César Augusto Sánchez Malaspina · **Periodo:** 2026-2 · **Modalidad:** individual
**Tema (registrado en PC1):** agente de revisión científica que identifica acuerdos y desacuerdos entre artículos.

---

### ¿Qué hace este proyecto, en palabras simples?

Imagina a un asistente que ayuda a revisar literatura científica. Le haces una pregunta como
*"¿Kaplan y Chinchilla están de acuerdo en cómo repartir el cómputo al entrenar un modelo?"* y debe:

1. **buscar** los fragmentos de los artículos que hablan del tema,
2. **contrastar** lo que dice cada artículo (¿coinciden o discrepan?),
3. **verificar** si una cita entre dos artículos es posible en el tiempo (un artículo de 2020 no puede citar a uno de 2022),
4. y **entregar** una respuesta ordenada con su evidencia, o **abstenerse** si no hay evidencia.

En el examen parcial ya construimos las piezas (el buscador, el modelo que redacta y la herramienta de citas).
En esta práctica respondemos dos preguntas nuevas:

* **¿Conviene dejar que un modelo de lenguaje decida los pasos (un *agente*) o basta con una receta fija (un *workflow*)?**
* **¿Ayuda que el agente *recuerde* lo que hizo antes, y qué pasa cuando ese recuerdo ya no es cierto?**

No buscamos el sistema "más autónomo", sino **medir** si la autonomía y la memoria mejoran algo de verdad.

### Mapa del cuaderno

| § | Qué se hace |
|---|---|
| 1 | Configuración y límites (fijados antes de experimentar) |
| 2 | Lo que se reutiliza del examen parcial (corpus, buscador, modelo, herramienta) |
| 3 | Herramientas y contratos |
| 4 | Controlador: quién decide qué, y cómo termina una ejecución |
| 5 | Benchmark de 12 tareas multi-paso |
| 6 | **Experimento 1:** workflow fijo (W0) frente a agente (W1) |
| 7 | **Experimento 2:** cuatro tipos de memoria (M0-M3) |
| 8 | Pruebas de robustez y terminación |
| 9 | **Demostración** para la defensa |
| 10 | Conclusiones |

### 1. Configuración

Todo se ejecuta en CPU con modelos locales y gratuitos. La primera celda fija `OMP_NUM_THREADS=1`
(en macOS, faiss y torch chocan si usan varios hilos; lo descubrimos en el parcial).

`RUN_EXPERIMENTS = True` vuelve a ejecutar los dos experimentos (**unos 40-60 minutos en CPU**).
Con `False`, el cuaderno carga los resultados ya guardados en `results/` y solo ejecuta la demostración.

In [1]:
import os, sys
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import json, time
import pandas as pd
pd.set_option("display.width", 180); pd.set_option("display.max_colwidth", 80); pd.set_option("display.max_columns", 30)

from baseline import *                      # componentes del parcial
from tools import ToolRuntime, EVIDENCE_SCORE_THRESHOLD, TOOL_NAMES
from controller import Limits, StopReason, run_task, ACTIONS
from workflow import DeterministicWorkflow
from agent import LLMAgent
from memory import make_memory
from evaluation import eval_exp1, eval_memory
from experiments import run_exp1, run_exp2

RUN_EXPERIMENTS = False   # True = repetir ambos experimentos (40-60 min)
LIMITS = Limits()   # mismos límites para W0, W1 y M0-M3

print("ROOT:", ROOT)
pd.Series({k: v for k, v in LIMITS.__dict__.items()}, name="límite").to_frame()

ROOT: /Users/cesar/Desktop/PC3-TOPICOS1-SanchezMalaspinaCesar


,límite
max_steps,8
max_tool_calls,6
total_deadline_s,180.0
timeout_per_tool_ms,"((search_evidence, 5000), (compare_evidence, 60000), (check_citation, 400))"
max_input_tokens,1800
max_memory_tokens,350
max_output_tokens_decision,40
max_output_tokens_args,32
max_output_tokens_id,16
max_output_tokens_compare,220


### 2. Lo que se reutiliza del examen parcial

El enunciado pide **conservar** el corpus, las consultas, los qrels, el recuperador, el modelo generador y la herramienta
con contratos. El código está en `src/baseline.py`, copiado sin cambios de comportamiento desde el cuaderno del parcial
(cada bloque indica su sección de origen). Los datos se copiaron al repositorio y **se verifica con las huellas SHA-256 del parcial**
que son exactamente los mismos.

| Componente | Origen | Uso en la PC3 |
|---|---|---|
| 36 abstracts de arXiv, 135 passages, IDs `arXivID-Pnn` | parcial §3 | evidencia de todas las tareas |
| 22 consultas, qrels y respuestas de referencia | parcial §3, §8 | resultados esperados del benchmark |
| Recuperación **C** (E5+FAISS, BM25, RRF k=60, Cross-Encoder) | parcial §6 | herramienta `search_evidence` |
| Qwen2.5-0.5B-Instruct, greedy, `repetition_penalty=1.0` | parcial §8 | analista de `compare_evidence` (igual en W0 y W1) |
| `citation.check_temporal_validity` con retry/timeout | parcial §10 | herramienta `check_citation` |

**Correcciones al parcial:** ninguna sobre sus resultados. Solo se agregó un filtro por fecha (`as_of`) **fuera** del recuperador,
para simular qué parte del corpus existía en cada sesión del experimento de memoria.

In [2]:
hashes = verify_same_corpus_as_parcial()
documents, queries, qrels, references, raw, passages_by_id = load_benchmark()
chunks = build_chunks(documents)
print("Mismo corpus que el parcial: OK")
print(f"{len(documents)} documentos · {len(passages_by_id)} passages · {len(chunks)} chunks · {len(queries)} consultas")

t0 = time.time()
retr = Retrievers(chunks)                                    # condición C del parcial
analyst = Generator(GENERATOR_MODEL_ID)                      # Qwen 0.5B: el MISMO analista para W0 y W1
policy_llm = Generator(AGENT_MODEL_ID, AGENT_MODEL_REVISION) # Qwen 1.5B: solo decide los pasos del agente W1
tools = ToolRuntime(retr, analyst, passages_by_id, raw)
print(f"Modelos cargados en {time.time()-t0:.0f} s · analista={analyst.model_id} · política del agente={policy_llm.model_id}")

Mismo corpus que el parcial: OK
36 documentos · 135 passages · 110 chunks · 22 consultas


W1112 12:41:24.536000 15257 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 21027.52it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 21412.55it/s]

... (18 outputs left out)

Loading weights:  93%|█████████▎| 315/338 [00:10<00:00, 36.23it/s]

Loading weights:  97%|█████████▋| 327/338 [00:10<00:00, 34.37it/s]

Loading weights: 100%|██████████| 338/338 [00:10<00:00, 30.74it/s]

Modelos cargados en 24 s · analista=Qwen/Qwen2.5-0.5B-Instruct · política del agente=Qwen/Qwen2.5-1.5B-Instruct


**¿Por qué el agente usa otro modelo para decidir?** Primero se probó Qwen 0.5B como cerebro del agente, solo con tareas de
**desarrollo** (`data/dev_tasks.jsonl`, temas que no están en la evaluación). En esas pruebas nunca eligió `check_citation`
aunque fuera lo único pendiente, y terminaba abandonando o repitiendo acciones (trazas en `results/dev/Qwen2.5-0.5B-Instruct.jsonl`).
Con Qwen 1.5B sí completó las tareas de desarrollo (`results/dev/Qwen2.5-1.5B-Instruct.jsonl`).

Para que la comparación W0/W1 sea justa, **el modelo que analiza la evidencia es el mismo en las dos variantes** (0.5B, el del parcial).
Lo único que cambia es *quién decide el siguiente paso*: reglas fijas (W0) o un LLM (W1).

In [3]:
dev_rows = []
for f in sorted((ROOT / "results" / "dev").glob("*.jsonl")):
    for r in read_jsonl(f):
        dev_rows.append({"modelo_politica": f.stem, "tarea_dev": r["task_id"], "stop_reason": r["stop_reason"],
                         "acciones": " → ".join(s["selected_tool"] or "·" for s in r["trace"])})
pd.DataFrame(dev_rows)

,modelo_politica,tarea_dev,stop_reason,acciones
0,Qwen2.5-0.5B-Instruct,D01,INVALID_OUTPUT,search_evidence → compare_evidence → compare_evidence → finish
1,Qwen2.5-0.5B-Instruct,D02,INVALID_OUTPUT,search_evidence → search_evidence → search_evidence → finish
2,Qwen2.5-0.5B-Instruct,D03,INVALID_OUTPUT,search_evidence → search_evidence → finish
3,Qwen2.5-0.5B-Instruct,DM1,ABSTAIN,search_evidence → compare_evidence → abstain
4,Qwen2.5-0.5B-Instruct,DM2,ABSTAIN,search_evidence → compare_evidence → compare_evidence → abstain
5,Qwen2.5-1.5B-Instruct,D01,SUCCESS,search_evidence → compare_evidence → compare_evidence → check_citation → finish
6,Qwen2.5-1.5B-Instruct,D02,SUCCESS,search_evidence → compare_evidence → compare_evidence → search_evidence → co...
7,Qwen2.5-1.5B-Instruct,D03,REPEATED_ACTION,search_evidence → compare_evidence → compare_evidence → compare_evidence
8,Qwen2.5-1.5B-Instruct,DM1,SUCCESS,search_evidence → compare_evidence → finish
9,Qwen2.5-1.5B-Instruct,DM2,SUCCESS,search_evidence → search_evidence → check_citation → finish


### 3. Herramientas y contratos

El modelo **nunca** ejecuta código ni comandos. Solo puede *proponer* una de cinco acciones de una lista cerrada
(`search_evidence`, `compare_evidence`, `check_citation`, `finish`, `abstain`). El controlador la ejecuta pasando por:

```text
nombre en la allowlist -> contrato de entrada (Pydantic) -> validación de dominio -> ejecución con timeout y reintentos
-> contrato de salida -> observación resumida como DATO (tipo y código del error, nunca su texto libre)
```

| Herramienta | Qué hace | Contrato de entrada | Errores posibles |
|---|---|---|---|
| `search_evidence` | busca en el corpus (condición C) hasta la fecha `as_of` | `query` de 8 a 300 caracteres | `validation`, `transient`, `timeout` |
| `compare_evidence` | el analista (Qwen 0.5B) dice si los artículos coinciden o discrepan | pregunta + 1 a 8 `passage_id` **ya observados** | `domain` (evidencia no observada), `output_contract` (JSON inválido) |
| `check_citation` | ¿la cita es posible por fechas? (herramienta del parcial) | dos arXiv ID con patrón `^\d{4}\.\d{4,5}$` | `validation`, `domain`, `transient`, `timeout`, `output_contract` |

**Umbral de evidencia.** `search_evidence` marca `evidence_found=True` si el mejor puntaje del Cross-Encoder es ≥ 2,0.
Se fijó con preguntas de desarrollo: las que no tienen respuesta en el corpus obtuvieron como máximo 1,24 y las que sí, al menos 2,52.

Prueba en vivo de cada herramienta, incluidos sus errores:

In [4]:
demo_calls = [
    ("search_evidence", {"query": "compute-optimal model size and training tokens"}, "none"),
    ("search_evidence", {"query": "hi"}, "none"),                                   # viola el contrato
    ("search_evidence", {"query": "compute-optimal model size"}, "transient_once"),  # falla 1 vez y se recupera
    ("check_citation", {"citing_id": "2203.15556", "cited_id": "2001.08361"}, "none"),
    ("check_citation", {"citing_id": "2001.08361", "cited_id": "2203.15556"}, "none"),  # veredicto negativo ≠ error
    ("check_citation", {"citing_id": "2203.15556", "cited_id": "9999.99999"}, "none"),  # fuera del catálogo
    ("check_citation", {"citing_id": "2203.15556", "cited_id": "2001.08361"}, "slow_always"),
    ("delete_files", {"path": "/"}, "none"),                                          # fuera de la allowlist
]
rows = []
for name, args, fault in demo_calls:
    o = tools.execute(name, args, fault=fault)
    rows.append({"herramienta": name, "argumentos": json.dumps(args), "fallo_inyectado": fault, "status": o.status,
                 "error": f"{o.error_kind}:{o.error_code}" if o.error_kind else "", "intentos": o.attempts,
                 "resultado": (f"{len(o.data['passages'])} passages, max_score={o.data['max_score']:.2f}" if name == "search_evidence" and o.data
                               else (f"temporally_valid={o.data['temporally_valid']}" if o.data else "")),
                 "ms": o.latency_ms})
pd.DataFrame(rows)

,herramienta,argumentos,fallo_inyectado,status,error,intentos,resultado,ms
0,search_evidence,"{""query"": ""compute-optimal model size and training tokens""}",none,ok,,1,"4 passages, max_score=10.16",1660.67
1,search_evidence,"{""query"": ""hi""}",none,error,validation:invalid_arguments,0,,0.24
2,search_evidence,"{""query"": ""compute-optimal model size""}",transient_once,ok,,2,"5 passages, max_score=5.65",256.93
3,check_citation,"{""citing_id"": ""2203.15556"", ""cited_id"": ""2001.08361""}",none,ok,,1,temporally_valid=True,1.76
4,check_citation,"{""citing_id"": ""2001.08361"", ""cited_id"": ""2203.15556""}",none,ok,,1,temporally_valid=False,0.66
5,check_citation,"{""citing_id"": ""2203.15556"", ""cited_id"": ""9999.99999""}",none,error,domain:unknown_paper,0,,0.50
6,check_citation,"{""citing_id"": ""2203.15556"", ""cited_id"": ""2001.08361""}",slow_always,error,timeout:attempt_timeout,3,,366.25
7,delete_files,"{""path"": ""/""}",none,error,protocol:unknown_tool,0,,0.00


### 4. Controlador: quién decide qué

| Lo decide **el LLM** (solo en W1) | Lo decide **el controlador determinista** (W0 y W1) |
|---|---|
| El pensamiento de cada paso (*Thought*), que queda en la traza | Si la acción propuesta se ejecuta o se bloquea |
| La siguiente acción, elegida entre 5 opciones permitidas | Contratos, reintentos y timeouts de cada herramienta |
| El texto de la búsqueda y los IDs de la cita | Qué entra al prompt (estado, historial, memoria) y su presupuesto de tokens |
| Cuándo terminar o abstenerse | La etiqueta de terminación (`stop_reason`) y la validación de la respuesta final |

**Ciclo del agente (ReAct):** `Perceive` (el controlador arma el estado) → `Decide` (Thought + Action) → `Act` (herramienta) →
`Observe` (observación validada) → actualizar estado → repetir o terminar.

La acción se elige con **decodificación restringida**: se calcula la probabilidad de cada una de las 5 acciones y se toma la mayor.
Así el modelo no puede inventar comandos, y las probabilidades quedan en la traza como justificación numérica de cada decisión.

**Estados de terminación** (definiciones verificables en `src/controller.py`):

| `stop_reason` | Cuándo ocurre |
|---|---|
| `SUCCESS` | se pidió `finish` y la respuesta final pasó la validación |
| `ABSTAIN` | la política decidió abstenerse o el análisis concluyó "evidencia insuficiente" |
| `MAX_STEPS` | se agotaron `max_steps` (8) o `max_tool_calls` (6) |
| `TIMEOUT` | se superó el plazo total de la tarea (180 s) |
| `TOOL_ERROR` | falta una salida requerida porque su herramienta falló sin recuperarse |
| `INVALID_OUTPUT` | dos decisiones inválidas seguidas, salida de herramienta que viola el contrato, o `finish` sin haber hecho lo requerido |
| `NO_PROGRESS` | 3 pasos seguidos sin actualizar el estado |
| `REPEATED_ACTION` | 2 intentos de repetir una acción idéntica ya ejecutada (la repetición no se ejecuta) |

Las pruebas del controlador usan políticas *scripted* (sin LLM) para forzar cada caso:

In [5]:
import subprocess
out = subprocess.run([sys.executable, "-m", "pytest", "-q", str(ROOT / "tests")], capture_output=True, text=True, cwd=ROOT)
print(out.stdout[-1500:])
assert out.returncode == 0

...................                                                      [100%]
19 passed in 0.20s



### 5. Benchmark de tareas (fijado antes de ejecutar)

12 tareas multi-paso, construidas con `src/build_data.py` a partir de las consultas y respuestas de referencia del parcial.
Los resultados esperados **no dependen del agente**: la relación correcta viene de `reference_answers.jsonl`,
la evidencia relevante de los qrels, y la validez de cada cita de las fechas de arXiv.

* **7 ordinarias** (T01-T07): buscar, contrastar y (en 6) verificar una cita. T04 y T07 piden **rechazar** una cita imposible por fechas.
* **3 con fallos inyectados** (T08-T10), identificados como tales: error transitorio de búsqueda, timeout persistente de la herramienta de citas, salida inválida del analista.
* **2 donde corresponde abstenerse** (T11-T12): la respuesta no está en el corpus (Mamba, Llama 3).

In [6]:
tasks = read_jsonl(DATA_DIR / "tasks.jsonl")
expected = {e["task_id"]: e for e in read_jsonl(DATA_DIR / "expected_results.jsonl")}
pd.DataFrame([{"task_id": t["task_id"], "categoría": t["categoria"], "pregunta": t["entrada"]["pregunta_es"],
               "operaciones": ", ".join(t["operaciones_necesarias"]),
               "fallo inyectado": "; ".join(f"{k}: {v}" for k, v in t["fallos_inyectados"].items()),
               "resultado esperado": t["resultado_esperado"], "terminación": t["condicion_de_terminacion"]} for t in tasks])

,task_id,categoría,pregunta,operaciones,fallo inyectado,resultado esperado,terminación
0,T01,ordinaria,¿Coinciden Kaplan y Chinchilla en cómo repartir un presupuesto de cómputo en...,"search_evidence, compare_evidence, check_citation",,relación=disagree; cita temporalmente válida=True,SUCCESS
1,T02,ordinaria,¿Las habilidades emergentes son un efecto real de la escala o un artefacto d...,"search_evidence, compare_evidence, check_citation",,relación=disagree; cita temporalmente válida=True,SUCCESS
2,T03,ordinaria,¿Compartir keys y values entre cabezas degrada la calidad frente a multi-hea...,"search_evidence, compare_evidence, check_citation",,relación=agree; cita temporalmente válida=True,SUCCESS
3,T04,rechazo_de_conclusion,¿DPO es superior a PPO (RLHF) para alinear modelos de lenguaje?,"search_evidence, compare_evidence, check_citation",,relación=disagree; cita temporalmente válida=False,SUCCESS
4,T05,ordinaria,¿La cuantización de pesos a 4 bits preserva el rendimiento de los LLM?,"search_evidence, compare_evidence",,relación=agree,SUCCESS
5,T06,ordinaria,"¿Los recuperadores densos superan a BM25, especialmente en zero-shot?","search_evidence, compare_evidence, check_citation",,relación=disagree; cita temporalmente válida=True,SUCCESS
6,T07,rechazo_de_conclusion,¿LoRA iguala la calidad del fine-tuning completo?,"search_evidence, compare_evidence, check_citation",,relación=disagree; cita temporalmente válida=False,SUCCESS
7,T08,fallo_herramienta,¿Cuantizar un LLM siempre acelera la inferencia?,"search_evidence, compare_evidence, check_citation",search_evidence: error transitorio en el primer intento (inyectado),relación=disagree; cita temporalmente válida=True,SUCCESS
8,T09,fallo_herramienta,¿El in-context learning requiere etiquetas correctas en las demostraciones?,"search_evidence, compare_evidence, check_citation",check_citation: timeout persistente en todos los intentos (inyectado),terminar con TOOL_ERROR sin inventar una respuesta,TOOL_ERROR
9,T10,fallo_herramienta,"¿Para inyectar conocimiento nuevo, RAG es mejor que el fine-tuning no superv...","search_evidence, compare_evidence",compare_evidence: salida que no cumple el contrato JSON (inyectada),terminar con INVALID_OUTPUT sin inventar una respuesta,INVALID_OUTPUT


### 6. Experimento 1 - Workflow determinista (W0) frente a agente controlado (W1)

| | |
|---|---|
| **Pregunta** | ¿Permitir que un LLM elija herramientas y decida el siguiente paso aumenta el éxito en tareas multi-paso respecto de un workflow fijo? |
| **Hipótesis** | En tareas "de receta" (siempre buscar → comparar → verificar), W1 no supera a W0 y cuesta más; podría ayudar solo si necesita reformular una búsqueda. |
| **Baseline** | W0: plan fijo con reglas (`src/workflow.py`) |
| **Variable modificada** | la política que controla la secuencia de acciones |
| **Variables fijas** | 12 tareas y resultados esperados, corpus `parcial-v1`, analista Qwen 0.5B greedy, recuperación C, herramientas y contratos, límites, sin memoria |

**Métricas.** `task_success` (todo correcto: terminación, relación, evidencia y cita), `termination_correct`, `steps`, `tool_calls`,
`tool_errors`, `latency_ms`, `tokens` (entrada + salida de todas las llamadas a LLM), `repeated_actions`.
Por ser ejecución local no se informa costo en dinero, solo tokens y tiempo.

In [7]:
workflow, agent = DeterministicWorkflow(), LLMAgent(policy_llm)
if RUN_EXPERIMENTS:
    t0 = time.time()
    per_task1, summary1, results1 = run_exp1([workflow, agent], tools, LIMITS)
    print(f"Experimento 1 ejecutado en {(time.time()-t0)/60:.1f} min")
else:
    per_task1 = pd.read_csv(RESULTS_DIR / "exp1" / "per_task.csv")
    summary1 = pd.read_csv(RESULTS_DIR / "exp1" / "summary.csv", index_col=0)
    results1 = read_jsonl(RESULTS_DIR / "exp1" / "results.jsonl")
summary1.T

policy,W0_workflow,W1_agent
n_tareas,12.000,12.000
task_success,0.500,0.083
termination_correct,0.750,0.167
relation_correct,0.250,0.125
evidence_supported,0.500,0.250
citation_correct,0.571,0.000
steps_media,3.167,3.583
tool_calls_media,2.167,2.583
tool_errors_total,4.000,5.000
repeated_actions_total,0.000,4.000


In [8]:
cols = ["task_id", "policy", "expected_stop", "stop_reason", "task_success", "termination_correct", "relation",
        "gold_relation", "evidence_supported", "citation_correct", "steps", "tool_calls", "tool_errors",
        "repeated_actions", "latency_ms", "tokens"]
per_task1[cols]

,task_id,policy,expected_stop,stop_reason,task_success,termination_correct,relation,gold_relation,evidence_supported,citation_correct,steps,tool_calls,tool_errors,repeated_actions,latency_ms,tokens
0,T01,W0_workflow,SUCCESS,ABSTAIN,False,False,NaN,disagree,False,False,2,1,0,0,268.9,0
1,T02,W0_workflow,SUCCESS,INVALID_OUTPUT,False,False,NaN,disagree,False,False,3,2,1,0,14139.4,885
2,T03,W0_workflow,SUCCESS,SUCCESS,True,True,agree,agree,True,True,4,3,0,0,11908.3,859
3,T04,W0_workflow,SUCCESS,INVALID_OUTPUT,False,False,NaN,disagree,False,False,3,2,1,0,14684.5,1030
4,T05,W0_workflow,SUCCESS,SUCCESS,True,True,agree,agree,True,NaN,3,2,0,0,7482.0,960
5,T06,W0_workflow,SUCCESS,SUCCESS,False,True,agree,disagree,True,True,4,3,0,0,5594.0,869
6,T07,W0_workflow,SUCCESS,SUCCESS,False,True,agree,disagree,True,True,4,3,0,0,4199.8,773
7,T08,W0_workflow,SUCCESS,SUCCESS,False,True,agree,disagree,False,True,4,3,0,0,4265.3,843
8,T09,W0_workflow,TOOL_ERROR,TOOL_ERROR,True,True,NaN,NaN,NaN,NaN,4,3,1,0,6143.9,1003
9,T10,W0_workflow,INVALID_OUTPUT,INVALID_OUTPUT,True,True,NaN,NaN,NaN,NaN,3,2,1,0,4510.5,833


**Comparación tarea por tarea.** Se buscan automáticamente (a) tareas donde W1 supera a W0 y (b) tareas donde W1 no mejora,
falla o cuesta más.

In [9]:
w = per_task1.pivot(index="task_id", columns="policy",
                    values=["task_success", "termination_correct", "stop_reason", "steps", "tokens", "latency_ms"])
cmp1 = pd.DataFrame({
    "W0_stop": w[("stop_reason", "W0_workflow")], "W1_stop": w[("stop_reason", "W1_agent")],
    "W0_ok": w[("task_success", "W0_workflow")], "W1_ok": w[("task_success", "W1_agent")],
    "W0_term_ok": w[("termination_correct", "W0_workflow")], "W1_term_ok": w[("termination_correct", "W1_agent")],
    "W0_steps": w[("steps", "W0_workflow")], "W1_steps": w[("steps", "W1_agent")],
    "W0_tokens": w[("tokens", "W0_workflow")], "W1_tokens": w[("tokens", "W1_agent")],
    "W1/W0_latencia": (w[("latency_ms", "W1_agent")] / w[("latency_ms", "W0_workflow")]).astype(float).round(1),
})
score = lambda r, p: int(r[f"{p}_ok"]) * 2 + int(r[f"{p}_term_ok"])
cmp1["veredicto"] = cmp1.apply(lambda r: "W1 mejor" if score(r, "W1") > score(r, "W0") else
                                ("W1 peor" if score(r, "W1") < score(r, "W0") else "igual"), axis=1)
cmp1

,W0_stop,W1_stop,W0_ok,W1_ok,W0_term_ok,W1_term_ok,W0_steps,W1_steps,W0_tokens,W1_tokens,W1/W0_latencia,veredicto
task_id,,,,,,,,,,,,
T01,ABSTAIN,SUCCESS,False,False,False,True,2,4,0,7226,390.0,W1 mejor
T02,INVALID_OUTPUT,REPEATED_ACTION,False,False,False,False,3,5,885,11711,7.2,igual
T03,SUCCESS,INVALID_OUTPUT,True,False,True,False,4,3,859,4662,3.2,W1 peor
T04,INVALID_OUTPUT,INVALID_OUTPUT,False,False,False,False,3,3,1030,4669,3.2,igual
T05,SUCCESS,SUCCESS,True,True,True,True,3,3,960,4289,4.4,igual
T06,SUCCESS,INVALID_OUTPUT,False,False,True,False,4,3,869,4623,6.6,W1 peor
T07,SUCCESS,INVALID_OUTPUT,False,False,True,False,4,3,773,4314,8.9,W1 peor
T08,SUCCESS,NO_PROGRESS,False,False,True,False,4,4,843,6935,13.9,W1 peor
T09,TOOL_ERROR,NO_PROGRESS,True,False,True,False,4,5,1003,10674,16.4,W1 peor


In [10]:
def show_trace(results, task_id, policy, max_chars=150):
    r = next(x for x in results if x["task_id"] == task_id and x["policy"] == policy)
    trace = r.get("trace") or [s for s in read_jsonl(RESULTS_DIR / "traces" / "exp1_traces.jsonl")
                               if s["task_id"] == task_id and s["policy"] == policy]
    print(f"== {policy} · {task_id} -> {r['stop_reason']} ({r['stop_detail']}) · pasos={r['steps']} · tokens={r['tokens']}")
    for s in trace:
        top = ""
        if s["action_scores"]:
            top = " | P(acción): " + ", ".join(f"{a}={p:.2f}" for a, p in sorted(s["action_scores"].items(), key=lambda x: -x[1])[:3])
        args = {k: v for k, v in s["arguments"].items() if k != "passage_ids"}
        print(f"  paso {s['step']}: {s['selected_tool']} {json.dumps(args)[:90]} -> {s['observation_status']} "
              f"{s['observation_error'] or ''}{top}")
        if s["thought"]:
            print(f"           Thought: {s['thought'][:max_chars]}")
        if s["stop_reason"]:
            print(f"           stop_reason={s['stop_reason']}: {s['note']}")
    fa = r["final_answer"]
    if fa:
        print(f"  respuesta: relation={fa['relation']} · evidencia={fa['evidence']} · cita={fa['citation_check'] and fa['citation_check'].get('temporally_valid')}")

better = cmp1[cmp1["veredicto"] == "W1 mejor"].index.tolist()
worse = cmp1[cmp1["veredicto"] == "W1 peor"].index.tolist()
print("W1 mejor en:", better or "ninguna", "· W1 peor en:", worse or "ninguna")
case_a = better[0] if better else cmp1.index[0]
case_b = worse[0] if worse else cmp1["W1/W0_latencia"].idxmax()
for tid in [case_a, case_b]:
    for pol in ["W0_workflow", "W1_agent"]:
        show_trace(results1, tid, pol)
    print()

W1 mejor en: ['T01'] · W1 peor en: ['T03', 'T06', 'T07', 'T08', 'T09', 'T10', 'T11', 'T12']
== W0_workflow · T01 -> ABSTAIN (la política decidió abstenerse) · pasos=2 · tokens=0
  paso 1: search_evidence {"query": "Do Kaplan-style scaling laws and the Chinchilla study agree on how to split a f -> ok 
           Thought: regla 1: buscar con la pregunta
  paso 2: abstain {} -> None 
           Thought: regla 2: max_score=0.53 < umbral -> abstenerse
           stop_reason=ABSTAIN: la política decidió abstenerse
== W1_agent · T01 -> SUCCESS (respuesta validada) · pasos=4 · tokens=7226
  paso 1: search_evidence {"query": "Kaplan-style scaling laws and Chinchilla study split compute budget model size  -> ok  | P(acción): search_evidence=1.00, compare_evidence=0.00, check_citation=0.00
           Thought: The task requires finding evidence, comparing it, and verifying a citation. Since no evidence has been found yet, the first step is to search for rele
  paso 2: compare_evidence {"question":

### 7. Experimento 2 - Estado y memoria del agente

| | |
|---|---|
| **Pregunta** | ¿Qué mejora aporta cada mecanismo de memoria a tareas encadenadas y qué errores introduce? |
| **Hipótesis** | La memoria ayuda cuando la tarea menciona algo hecho antes ("el artículo de la revisión anterior"); la episódica puede devolver recuerdos viejos; la semántica con fechas y estado debería evitar usar un hecho superado. |
| **Baseline** | M0: el agente sin memoria |
| **Variable modificada** | el mecanismo de memoria |
| **Variables fijas** | agente W1, analista, herramientas, controlador, límites, 11 interacciones y **el mismo presupuesto de memoria (350 tokens)** |

**Qué es cada memoria, operacionalmente** (`src/memory.py`):

| Config. | Qué guarda | Dónde | Cuánto dura | Cómo se recupera |
|---|---|---|---|---|
| **M0** sin memoria | nada | — | — | — |
| **M1** de trabajo | resumen de cada interacción anterior **de la sesión** | RAM | se borra al empezar otra sesión | todas, de la más reciente a la más antigua, hasta el presupuesto |
| **M2** episódica | episodio completo: tarea, acciones, documentos observados, resultado, errores | JSONL en disco | entre sesiones | top-2 por relevancia (E5) + recencia + importancia (Park et al., 2023) |
| **M3** semántica | un **hecho** por artículo validado: afirmación, `passage_id`, versión arXiv, fecha del corpus, estado | JSONL + índice FAISS | entre sesiones | top-3 por similitud E5 + FAISS (Semana 4) |

**Escritura:** la hace el controlador al terminar cada interacción, no el LLM. **Actualización (M3):** si un artículo vuelve a usarse, sube su `version`.
**Invalidación (M3):** si una revisión hecha con un corpus más nuevo cita un artículo del mismo tema publicado *después* de la fecha de un hecho anterior,
ese hecho pasa a `SUPERSEDED` (no se borra; queda como historia marcada). **Eliminación:** `delete(memory_id)` (probada en `tests/test_memory.py`).

**Escenarios** (`data/memory_scenarios.jsonl`): 3 secuencias, 11 interacciones, 6 sesiones. Para simular "publicaciones nuevas" sin inventar datos,
cada sesión solo ve los artículos publicados hasta su fecha (`as_of`); al pasar de 2021 a 2023 **aparecen** artículos reales que ya estaban en el corpus.
Esto es una **actualización controlada del escenario**, declarada en los datos.

In [11]:
scen = read_jsonl(DATA_DIR / "memory_scenarios.jsonl")
pd.DataFrame([{"id": s["task_id"], "sesión": s["session_id"], "corpus hasta": s["as_of"], "tipo": s["task_type"],
               "entrada": s["input_en"][:110], "necesita memoria": s["memory_role"]["needs_memory"],
               "otra sesión": s["memory_role"].get("cross_session", False),
               "recuerdo correcto": s["memory_role"].get("gold_doc_id", ""),
               "recuerdo que interfiere / obsoleto": ", ".join(s["memory_role"].get("interference_doc_ids", []) +
                                                               ([s["memory_role"]["stale_doc_id"]] if s["memory_role"].get("stale_doc_id") else [])),
               "actualización del escenario": "sí" if s["scenario_update"] else ""} for s in scen])

,id,sesión,corpus hasta,tipo,entrada,necesita memoria,otra sesión,recuerdo correcto,recuerdo que interfiere / obsoleto,actualización del escenario
0,S1.1,S1-A,2021-12-31,review,Question: How should a fixed compute budget be split between model size and ...,False,False,,,
1,S1.2,S1-A,2021-12-31,verify_citation,Citing paper: 2005.11401. Cited paper: the paper that supported the compute-...,True,False,2001.08361,,
2,S1.3,S1-B,2023-12-31,review,Question: How should a fixed compute budget be split between model size and ...,False,False,,,sí
3,S1.4,S1-B,2023-12-31,verify_citation,Citing paper: 2305.14314. Cited paper: the paper with the most recent (curre...,True,False,2203.15556,2001.08361,
4,S2.1,S2-A,2022-12-31,review,Question: Are emergent abilities of large language models a real effect of s...,False,False,,,
5,S2.2,S2-A,2022-12-31,review,Question: Does 8-bit quantization preserve the performance of large transfor...,False,False,,,
6,S2.3,S2-B,2023-12-31,verify_citation,Citing paper: 2304.15004. Cited paper: the paper on emergent abilities of la...,True,True,2206.07682,2208.07339,
7,S2.4,S2-B,2023-12-31,verify_citation,Citing paper: 2210.17323. Cited paper: the 8-bit quantization paper recorded...,True,True,2208.07339,2206.07682,
8,S3.1,S3-A,2023-12-31,review,Question: Does sharing keys and values across attention heads degrade model ...,False,False,,,
9,S3.2,S3-A,2023-12-31,verify_citation,Citing paper: 2305.13245. Cited paper: the paper that introduced multi-query...,True,False,1911.02150,2305.13245,


**Métricas y denominadores**

| Métrica | Numerador | Denominador |
|---|---|---|
| `task_success` | interacciones resueltas (revisión: termina en SUCCESS citando evidencia esperada; cita: par y veredicto correctos) | 11 |
| `memory_recall_accuracy` | el recuerdo correcto **estaba en el contexto**, el agente **usó su ID** y **no** lo obtuvo de una búsqueda en esa misma interacción | 7 interacciones que necesitan memoria |
| `cross_session_retention` | igual, pero el recuerdo viene de **otra sesión** | 3 |
| `stale_memory_error_rate` | el agente usó el ID de un recuerdo obsoleto | interacciones con un recuerdo obsoleto en el contexto |
| `wrong_memory_use_rate` | el agente usó el ID de un recuerdo que no correspondía | interacciones con al menos un recuerdo en el contexto |
| `context_tokens` | tokens de entrada de todas las llamadas al LLM | por interacción |

Una respuesta correcta obtenida **buscando en el corpus** se cuenta como `solved_by_tools`, no como recuperación de memoria.

In [12]:
if RUN_EXPERIMENTS:
    t0 = time.time()
    per2, summary2, results2 = run_exp2(agent, tools, retr, policy_llm, documents, passages_by_id, LIMITS)
    print(f"Experimento 2 ejecutado en {(time.time()-t0)/60:.1f} min")
else:
    per2 = pd.read_csv(RESULTS_DIR / "exp2" / "per_interaction.csv", dtype={"cited_id_proposed": str, "gold_doc_id": str})
    summary2 = pd.read_csv(RESULTS_DIR / "exp2" / "summary.csv", index_col=0)
    results2 = read_jsonl(RESULTS_DIR / "exp2" / "results.jsonl")
summary2

,n,task_success,memory_recall_accuracy,recall_n,cross_session_retention,cross_n,stale_memory_error_rate,stale_n,wrong_memory_use_rate,wrong_n,solved_by_tools,context_tokens_media,memory_tokens_media,steps_media,latency_ms_media
config,,,,,,,,,,,,,,,
M0,11,0.182,0.0,0/6,0.0,0/3,NaN,0/0,NaN,0/0,0/6,7934.000,0.000,4.818,68623.873
M1,11,0.182,0.0,0/6,0.0,0/3,0.0,0/1,0.0,0/5,1/6,9361.727,92.091,4.818,67443.200
M2,11,0.273,0.0,0/6,0.0,0/3,0.0,0/1,0.0,0/10,0/6,10686.545,228.636,4.545,75887.227
M3,11,0.273,0.0,0/6,0.0,0/3,NaN,0/0,0.0,0/6,1/6,10808.455,154.182,4.818,74598.136


In [13]:
cols2 = ["config", "task_id", "session_id", "task_type", "stop_reason", "task_success", "needs_memory", "cross_session",
         "memory_ids_in_context", "cited_id_proposed", "gold_doc_id", "gold_memory_in_context", "memory_recall_correct",
         "solved_by_tools", "wrong_memory_use", "stale_in_context", "stale_memory_error", "memory_tokens", "context_tokens", "steps"]
per2[[c for c in cols2 if c in per2.columns]]

,config,task_id,session_id,task_type,stop_reason,task_success,needs_memory,cross_session,memory_ids_in_context,cited_id_proposed,gold_doc_id,gold_memory_in_context,memory_recall_correct,solved_by_tools,wrong_memory_use,stale_in_context,stale_memory_error,memory_tokens,context_tokens,steps
0,M0,S1.1,S1-A,review,REPEATED_ACTION,False,False,False,[],NaN,NaN,NaN,NaN,NaN,False,False,False,0,4317,3
1,M0,S1.2,S1-A,verify_citation,MAX_STEPS,False,True,False,[],NaN,2001.08361,False,False,False,False,False,False,0,12708,7
2,M0,S1.3,S1-B,review,NO_PROGRESS,False,False,False,[],NaN,NaN,NaN,NaN,NaN,False,False,False,0,6087,4
3,M0,S1.4,S1-B,verify_citation,REPEATED_ACTION,False,True,False,[],NaN,2203.15556,False,False,False,False,False,False,0,6090,4
4,M0,S2.1,S2-A,review,SUCCESS,True,False,False,[],NaN,NaN,NaN,NaN,NaN,False,False,False,0,4302,3
5,M0,S2.2,S2-A,review,SUCCESS,True,False,False,[],NaN,NaN,NaN,NaN,NaN,False,False,False,0,4113,3
6,M0,S2.3,S2-B,verify_citation,SUCCESS,False,True,True,[],1706.03762,2206.07682,False,False,False,False,False,False,0,11189,6
7,M0,S2.4,S2-B,verify_citation,REPEATED_ACTION,False,True,True,[],NaN,2208.07339,False,False,False,False,False,False,0,11104,6
8,M0,S3.1,S3-A,review,REPEATED_ACTION,False,False,False,[],NaN,NaN,NaN,NaN,NaN,False,False,False,0,11144,7
9,M0,S3.2,S3-A,verify_citation,NO_PROGRESS,False,True,False,[],NaN,1911.02150,False,False,False,False,False,False,0,10031,6


In [14]:
grid = per2.pivot(index="task_id", columns="config", values="task_success").astype(int)
grid["M0→mejor"] = grid[["M1", "M2", "M3"]].max(axis=1) - grid["M0"]
print("1 = interacción resuelta")
grid

1 = interacción resuelta


config,M0,M1,M2,M3,M0→mejor
task_id,,,,,
S1.1,0,0,0,0,0
S1.2,0,0,0,0,0
S1.3,0,0,1,0,1
S1.4,0,0,0,0,0
S2.1,1,1,1,1,0
S2.2,1,0,1,1,0
S2.3,0,0,0,1,1
S2.4,0,0,0,0,0
S3.1,0,0,0,0,0


**Qué quedó guardado en la memoria persistente** (estado final de los almacenes en disco):

In [15]:
store = RESULTS_DIR / "exp2" / "memory_store"
for name in ["M2_episodic", "M3_semantic"]:
    items = read_jsonl(store / f"{name}.jsonl") if (store / f"{name}.jsonl").exists() else []
    print(f"--- {name}: {len(items)} recuerdos")
    for it in items:
        extra = f" | SUPERSEDED by {it['superseded_by']}" if it.get("status") == "superseded" else ""
        ver = f" v{it['version']}" if "version" in it else ""
        print(f"  {it['memory_id']}{ver} [sesión {it['session_id']}, corpus {it['as_of']}{extra}] {it['text'][:150]}")

--- M2_episodic: 11 recuerdos
  E-S1.1 [sesión S1-A, corpus 2021-12-31] Task S1.1 (Review what the available papers recommend for splitting a compute budget.) ended REPEATED_ACTION. Papers observed (not validated): 2001.08
  E-S1.2 [sesión S1-A, corpus 2021-12-31] Task S1.2 (Verify whether a citation is temporally possible.) ended REPEATED_ACTION. Papers observed (not validated): 2001.08361 "Scaling Laws for Neu
  E-S1.3 [sesión S1-B, corpus 2023-12-31] Task S1.3 (Review again, with today's corpus, the compute-budget recommendation recorded earlier.) ended SUCCESS. Supporting papers (validated): 2203.
  E-S1.4 [sesión S1-B, corpus 2023-12-31] Task S1.4 (Verify whether a citation is temporally possible.) ended NO_PROGRESS. Papers observed (not validated): 1911.02150 "Fast Transformer Decodin
  E-S2.1 [sesión S2-A, corpus 2022-12-31] Task S2.1 (Review what the available papers say about emergent abilities.) ended SUCCESS. Supporting papers (validated): 2206.07682 "Emergent Abilitie
  E-S

**Análisis obligatorio:** un caso donde la memoria ayuda, uno donde no mejora y uno donde recupera información equivocada,
obsoleta o irrelevante. Se seleccionan desde los resultados, no a mano.

In [16]:
def show_mem_case(cfg, tid):
    r = next(x for x in results2 if x["config"] == cfg and x["task_id"] == tid)
    row = per2[(per2.config == cfg) & (per2.task_id == tid)].iloc[0]
    print(f"== {cfg} · {tid} -> {r['stop_reason']} · éxito={row['task_success']} · recuerdos en contexto={r['memory_ids_in_context']}")
    for m in r.get("memory_items", []):
        print(f"   recuerdo {m['memory_id']} ({m['kind']}, sesión {m['session_id']}, corpus {m['as_of']}, estado {m['status']}) docs={m['doc_ids']}")
    trace = r.get("trace") or [s for s in read_jsonl(RESULTS_DIR / "traces" / "exp2_traces.jsonl")
                               if s["task_id"] == tid and s["run"] == f"exp2/{cfg}"]
    for s in trace:
        args = {k: v for k, v in s["arguments"].items() if k != "passage_ids"}
        print(f"   paso {s['step']}: {s['selected_tool']} {json.dumps(args)[:80]} -> {s['observation_status']} {s['observation_error'] or ''}")
        if s["thought"]:
            print(f"            Thought: {s['thought'][:140]}")
    print()

# Métrica estricta (definida antes de ejecutar): el recuerdo se usó Y la búsqueda no encontró el documento.
helped_strict = per2[(per2.config != "M0") & per2.memory_recall_correct.fillna(False).astype(bool)]
# Si no hubo ninguno, se usa un criterio descriptivo para ELEGIR el caso (no cambia la métrica):
# con memoria se resolvió una interacción que M0 no resolvió y el documento correcto estaba en la memoria.
m0_ok = set(per2[(per2.config == "M0") & per2.task_success.astype(bool)].task_id)
helped = helped_strict if len(helped_strict) else per2[(per2.config != "M0") & per2.task_success.astype(bool)
                                                       & per2.gold_memory_in_context.fillna(False).astype(bool)
                                                       & ~per2.task_id.isin(m0_ok)]
wrong = per2[per2.wrong_memory_use.astype(bool) | per2.stale_memory_error.astype(bool)]
if not len(wrong):  # interferencia: citó un documento que estaba en un recuerdo que NO correspondía
    def _interf(r):
        mem = next((x for x in results2 if x["config"] == r.config and x["task_id"] == r.task_id), {})
        docs = {d for m in mem.get("memory_items", []) for d in (m.get("doc_ids") or [])}
        return bool(r.needs_memory) and isinstance(r.cited_id_proposed, str) and r.cited_id_proposed in docs \
            and r.cited_id_proposed != str(r.gold_doc_id)
    wrong = per2[[_interf(r) for r in per2.itertuples()]]
print(f"métrica estricta memory_recall_correct: {len(helped_strict)} casos (ver README §8)")
no_gain = per2[(per2.config != "M0") & per2.needs_memory.astype(bool) & ~per2.memory_recall_correct.fillna(False).astype(bool)
               & per2.gold_memory_in_context.fillna(False).astype(bool)]
print(f"recuerdo usado correctamente: {len(helped)} · recuerdo equivocado u obsoleto usado: {len(wrong)} · "
      f"recuerdo correcto disponible pero no usado: {len(no_gain)}\n")
for label, df in [("(a) la memoria ayuda", helped), ("(b) la memoria no mejora", no_gain), ("(c) recuerdo equivocado/obsoleto", wrong)]:
    print("#####", label)
    if len(df):
        show_mem_case(df.iloc[0]["config"], df.iloc[0]["task_id"])
    else:
        print("   no ocurrió en esta ejecución\n")

métrica estricta memory_recall_correct: 0 casos (ver README §8)
recuerdo usado correctamente: 2 · recuerdo equivocado u obsoleto usado: 2 · recuerdo correcto disponible pero no usado: 10

##### (a) la memoria ayuda
== M1 · S3.2 -> SUCCESS · éxito=True · recuerdos en contexto=['W-S3.1']
   recuerdo W-S3.1 (working, sesión S3-A, corpus 2023-12-31, estado active) docs=['1911.02150', '2305.13245', '2208.07339']
   paso 1: search_evidence {"query": "arXiv:2305.13245"} -> ok 
            Thought: The task requires verifying the temporal possibility of a citation. Since no evidence has been found yet, I need to search for relevant pape
   paso 2: check_citation {"citing_id": "2305.13245", "cited_id": "1911.02150"} -> ok 
            Thought: The task requires verifying the temporal possibility of a citation. I need to check if the citing paper (2305.13245) can cite the cited pape
   paso 3: search_evidence {"query": "papers on multi-query attention (one shared key-value head)"} -> ok 
       

### 8. Pruebas de robustez y terminación

El enunciado pide seis pruebas controladas. Cada una tiene evidencia en las trazas reales de los experimentos y, además,
una prueba unitaria con política *scripted* (sección 4).

In [17]:
tr1 = read_jsonl(RESULTS_DIR / "traces" / "exp1_traces.jsonl")
tr2 = read_jsonl(RESULTS_DIR / "traces" / "exp2_traces.jsonl")
def stops(task_id):
    return ", ".join(f"{r['policy'][:2]}={r['stop_reason']}" for r in results1 if r["task_id"] == task_id)
stale_rows = per2[per2.stale_in_context.astype(bool)]
robust = pd.DataFrame([
    {"prueba": "error transitorio", "dónde": "T08 (search falla 1 vez)", "observado": stops("T08"),
     "reintentos en traza": sum(s["attempts"] > 1 for s in tr1 if s["task_id"] == "T08"), "unit test": "test_transient_error_recorded_in_trace"},
    {"prueba": "salida inválida", "dónde": "T10 (analista devuelve texto no JSON)", "observado": stops("T10"),
     "reintentos en traza": "", "unit test": "test_invalid_tool_output"},
    {"prueba": "acción repetida sin avance", "dónde": "todas las ejecuciones de W1",
     "observado": f"{sum(1 for s in tr1 + tr2 if s['note'] == 'repetición bloqueada' or s['stop_reason'] == 'REPEATED_ACTION')} repeticiones bloqueadas",
     "reintentos en traza": "", "unit test": "test_repeated_action_is_blocked_then_stops"},
    {"prueba": "agotamiento de pasos", "dónde": "límite max_steps=8 / max_tool_calls=6",
     "observado": f"{sum(r['stop_reason'] == 'MAX_STEPS' for r in results1 + results2)} ejecuciones con MAX_STEPS",
     "reintentos en traza": "", "unit test": "test_max_steps, test_max_tool_calls_maps_to_max_steps"},
    {"prueba": "evidencia insuficiente", "dónde": "T11, T12 (Mamba, Llama 3)", "observado": stops("T11") + " | " + stops("T12"),
     "reintentos en traza": "", "unit test": "test_abstain"},
    {"prueba": "recuerdo obsoleto vs. fuente más reciente", "dónde": "S1.4 (Kaplan superado por Chinchilla)",
     "observado": "; ".join(f"{r.config}: citó {r.cited_id_proposed}" for r in per2[per2.task_id == "S1.4"].itertuples()),
     "reintentos en traza": "", "unit test": "test_semantic_invalidation_update_and_delete"},
])
robust

,prueba,dónde,observado,reintentos en traza,unit test
0,error transitorio,T08 (search falla 1 vez),"W0=SUCCESS, W1=NO_PROGRESS",3,test_transient_error_recorded_in_trace
1,salida inválida,T10 (analista devuelve texto no JSON),"W0=INVALID_OUTPUT, W1=NO_PROGRESS",,test_invalid_tool_output
2,acción repetida sin avance,todas las ejecuciones de W1,33 repeticiones bloqueadas,,test_repeated_action_is_blocked_then_stops
3,agotamiento de pasos,límite max_steps=8 / max_tool_calls=6,4 ejecuciones con MAX_STEPS,,"test_max_steps, test_max_tool_calls_maps_to_max_steps"
4,evidencia insuficiente,"T11, T12 (Mamba, Llama 3)","W0=ABSTAIN, W1=SUCCESS | W0=ABSTAIN, W1=SUCCESS",,test_abstain
5,recuerdo obsoleto vs. fuente más reciente,S1.4 (Kaplan superado por Chinchilla),M0: citó nan; M1: citó nan; M2: citó nan; M3: citó nan,,test_semantic_invalidation_update_and_delete


In [18]:
# Ningún error se convirtió en una respuesta aparentemente válida:
bad = [r for r in results1 + results2 if r["stop_reason"] != "SUCCESS" and r["final_answer"] is not None
       and r["stop_reason"] != "ABSTAIN"]
assert not bad, bad
fabricated = [r["task_id"] for r in results1 if r["task_id"] in ("T09", "T10") and r["stop_reason"] == "SUCCESS"]
print("Ejecuciones no exitosas con respuesta fabricada:", len(bad), "· tareas con fallo persistente que terminaron en SUCCESS:", fabricated or "ninguna")
print("Pasos registrados en trazas:", len(tr1), "(exp1) +", len(tr2), "(exp2)")

Ejecuciones no exitosas con respuesta fabricada: 0 · tareas con fallo persistente que terminaron en SUCCESS: ninguna
Pasos registrados en trazas: 81 (exp1) + 209 (exp2)


### 9. Demostración

Una sola tarea multi-paso ejecutada por el **workflow** y por el **agente**, con una decisión y una observación registradas,
una recuperación de memoria, un error controlado, una métrica y la condición de terminación.
Cambia `DEMO_TASK` para probar otra tarea del benchmark.

In [19]:
DEMO_TASK = "T03"
task = next(t for t in tasks if t["task_id"] == DEMO_TASK)
print("[1] Tarea:", task["entrada"]["pregunta_es"], "| cita a verificar:", task["entrada"]["cita_a_verificar"])
for pol in [workflow, agent]:
    r = run_task(task, pol, tools, LIMITS, run_label="demo")
    print(f"\n[2] {pol.name}: stop_reason={r['stop_reason']} · pasos={r['steps']} · llamadas={r['tool_calls']} · "
          f"tokens={r['tokens']} · {r['latency_ms']/1000:.1f} s")
    for s in r["trace"]:
        p = f" P={max(s['action_scores'].values()):.2f}" if s["action_scores"] else ""
        print(f"    paso {s['step']}: Thought='{s['thought'][:70]}' -> Action={s['selected_tool']}{p} -> obs={s['observation_status']}")
    m = eval_exp1(r, expected[DEMO_TASK])
    print(f"[3] métrica: task_success={m['task_success']} · relation={m['relation']} (gold {m['gold_relation']}) · "
          f"evidencia con qrel={m['evidence_supported']} · cita correcta={m['citation_correct']}")

[1] Tarea: ¿Compartir keys y values entre cabezas degrada la calidad frente a multi-head attention? | cita a verificar: {'citing_id': '2305.13245', 'cited_id': '1911.02150'}



[2] W0_workflow: stop_reason=SUCCESS · pasos=4 · llamadas=3 · tokens=859 · 14.2 s
    paso 1: Thought='regla 1: buscar con la pregunta' -> Action=search_evidence -> obs=ok
    paso 2: Thought='regla 3: analizar la evidencia recuperada' -> Action=compare_evidence -> obs=ok
    paso 3: Thought='regla 4: verificar la cita indicada en la tarea' -> Action=check_citation -> obs=ok
    paso 4: Thought='regla 5: todas las salidas requeridas están disponibles' -> Action=finish -> obs=None
[3] métrica: task_success=True · relation=agree (gold agree) · evidencia con qrel=True · cita correcta=True



[2] W1_agent: stop_reason=INVALID_OUTPUT · pasos=3 · llamadas=2 · tokens=4662 · 90.5 s
    paso 1: Thought='I need to search for papers related to the question and citation to fi' -> Action=search_evidence P=1.00 -> obs=ok
    paso 2: Thought='The task requires comparing the evidence found with the citation to ve' -> Action=compare_evidence P=1.00 -> obs=ok
    paso 3: Thought='The evidence is already found and agreed upon. No further action is ne' -> Action=finish P=1.00 -> obs=None
[3] métrica: task_success=False · relation=None (gold agree) · evidencia con qrel=False · cita correcta=False


In [20]:
# [4] Error controlado: la herramienta de citas no responde (timeout persistente)
r = run_task({**task, "task_id": "DEMO-ERR", "faults": {"check_citation": "slow_always"}}, workflow, tools, LIMITS, run_label="demo")
s = next((s for s in r["trace"] if s["selected_tool"] == "check_citation"), None)
if s is None:
    raise RuntimeError(f"{DEMO_TASK}: el workflow terminó en {r['stop_reason']} antes de verificar la cita; elige otra DEMO_TASK")
print(f"[4] check_citation -> {s['observation_error']} tras {s['attempts']} intentos · stop_reason={r['stop_reason']} "
      f"· respuesta final={r['final_answer']}  (no se inventa el veredicto)")

# [5] Recuperación de memoria: revisión en la sesión A (corpus 2021) -> verificación en la sesión B
import tempfile
demo_mem = make_memory("M3", store_dir=Path(tempfile.mkdtemp()), passages_by_id=passages_by_id,
                       count_tokens=policy_llm.count_tokens, encoder=retr, documents=documents)
s1, s4 = next(x for x in scen if x["task_id"] == "S2.1"), next(x for x in scen if x["task_id"] == "S2.3")
demo_mem.start_session("DEMO-A", s1["as_of"])
r1 = run_task(s1, workflow, tools, LIMITS, memory=demo_mem, run_label="demo")   # W0 solo para poblar la memoria rápido
print(f"[5] sesión A: {s1['task_id']} terminó en {r1['stop_reason']} -> hechos escritos: {[f['memory_id'] for f in demo_mem.write(s1, r1)]}")
demo_mem.start_session("DEMO-B", s4["as_of"])
block, items, ntok = demo_mem.context_for(s4, LIMITS.max_memory_tokens)
print(f"\n[5] Memoria semántica recuperada para '{s4['input_en']}' ({ntok} tokens):\n{block}")

[4] check_citation -> timeout:attempt_timeout tras 3 intentos · stop_reason=TOOL_ERROR · respuesta final=None  (no se inventa el veredicto)


[5] sesión A: S2.1 terminó en SUCCESS -> hechos escritos: ['F-2206.07682', 'F-2208.07339']

[5] Memoria semántica recuperada para 'Citing paper: 2304.15004. Cited paper: the paper on emergent abilities of large language models (not the quantization paper) recorded in an earlier session.' (196 tokens):
[F-2206.07682 | semantic | session DEMO-A | corpus as of 2022-12-31] 2206.07682 "Emergent Abilities of Large Language Models" (published 2022-06-15): Scaling up language models has been shown to predictably improve performance and sample efficiency on a wide range of downstream tasks. This ...
[F-2208.07339 | semantic | session DEMO-A | corpus as of 2022-12-31] 2208.07339 "LLM.int8(): 8-bit Matrix Multiplication for Transformers at Scale" (published 2022-08-15): Large language models have been widely adopted but require significant GPU memory for inference. We develop a procedure for Int8 matrix multiplication ...


### 10. Conclusiones

Números de la ejecución oficial guardada en `results/` (una sola corrida, decodificación greedy, 12 tareas y 11 interacciones por configuración: con muestras tan pequeñas, una diferencia de una tarea **no** es significativa).

**Experimento 1: la receta fija le gana al agente en este problema.**

| | W0 workflow | W1 agente |
|---|---|---|
| Tareas resueltas | **6/12** | 1/12 |
| Terminó con la etiqueta correcta | **9/12** | 2/12 |
| Tokens por tarea | **671** | 6 222 (≈ 9×) |
| Tiempo por tarea | **6,1 s** | 59,1 s (≈ 10×) |
| Acciones repetidas bloqueadas | 0 | 4 |

* **Dónde gana W1 (T01).** La pregunta literal de W0 no pasó el umbral de evidencia (0,53 < 2,0) y W0 se abstuvo por error. El agente reformuló la búsqueda, encontró a Kaplan y Chinchilla y terminó en `SUCCESS`, que era el cierre esperado. Aun así, la tarea no cuenta como resuelta: invirtió el orden de la cita y el analista se equivocó en la relación.
* **Dónde no gana W1 (T03, T06, T07).** W1 llamó a `finish` sin verificar la cita, aunque la lista de salidas la marcaba como PENDIENTE. El controlador no aceptó esa respuesta (`INVALID_OUTPUT`). En T11 y T12, W1 respondió `agree` con evidencia por debajo del umbral, mientras que W0 se abstuvo correctamente.
* **Errores.** Con fallos inyectados, W0 terminó con la etiqueta exacta (`TOOL_ERROR` en T09, `INVALID_OUTPUT` en T10). W1 terminó en `NO_PROGRESS`: no mintió, pero no reconoció el error. Ninguna ejecución con error devolvió una respuesta inventada (verificado en §8).

**Experimento 2: la memoria ayudó poco y también metió errores.**

* `task_success`: M0 2/11, M1 2/11, M2 3/11, M3 3/11. La memoria aumentó el contexto entre un 18 % y un 36 % (7 934 tokens con M0 frente a 10 808 con M3).
* **Ayuda (M3, S2.3):** el hecho semántico de *Emergent Abilities* venía de otra sesión. El agente verificó la cita correcta, mientras que M0 y M1 citaron un artículo equivocado (1706.03762). La búsqueda también encontró el documento, así que la métrica estricta `memory_recall_accuracy` lo cuenta como resuelto por las herramientas: 0/6 en todas las configuraciones.
* **No mejora (S1.2, S1.4, S3.3):** el recuerdo correcto estaba en el contexto y el agente igualmente se trabó (`NO_PROGRESS`, `REPEATED_ACTION`).
* **Recuerdo que confunde (M2, S2.3):** con el episodio de LLM.int8 en el contexto, el agente mezcló los dos temas en la búsqueda y citó 2208.07339 en lugar de 2206.07682. Es la interferencia que el escenario buscaba provocar.
* La invalidación (`SUPERSEDED`) funciona en las pruebas unitarias. En la evaluación no llegó a activarse porque el agente no resolvió S1.1 ni S1.3 con M3, y la memoria semántica solo guarda tareas exitosas.

**Decisión técnica.** Para este problema conviene la **receta fija con el LLM solo como analista**. Es entre 9 y 10 veces más barata y termina bien el triple de veces. El agente solo aporta cuando hay que reformular la búsqueda (T01). Una mejora barata sería un W0 que reformule la búsqueda una sola vez antes de abstenerse. Si se usa memoria, la recomendación es la semántica con estado y fuente (M3) y la validación del controlador, nunca recuerdos en texto libre (M2), que fueron los que provocaron la interferencia.